# Задание 4: объединение лучших идей — вероятностная голова и preprocessing

Это отдельная копия исходного решения; `task4.ipynb` не изменяется. Исследуются 12 представлений признаков с зафиксированной сильной сетью из эксперимента функций потерь: два Linear/BatchNorm/ReLU блока ширины 128, dropout0.55, AdamW weight_decay0.1, распределение года по 91 значению, мягкая разметка sigma0.6, функция потерь normalized MSE +2×cross entropy. Предсказание — математическое ожидание года одной сети, без ансамбля.

Данные только из предоставленных `train_x.csv`, `train_y.csv`, `test_x.csv`. Сохраняем исходный protocol9800/2100/2100, seeds42/17/83, статистики train-only. Во время подбора audit закрыт; диагностическая проверка добавлена только после фиксации всех кандидатов. После единственного этапа взаимодействий три лидера фиксируются для подтверждения. Новых настроек по подтверждению не добавляется.

Код всех преобразований, модели, собственного оптимизатора, обучения и сборки CSV находится в этом файле. Другие ноутбуки и временные файлы при запуске не читаются.


In [1]:
import base64
import copy
import hashlib
import io
import math
import random
import time
import unittest
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn
import torch
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import QuantileTransformer, RobustScaler, StandardScaler
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
SELECTION_METRIC = "rmse"  # "rmse", "mse" или "mae"; см. пояснение выше.
assert SELECTION_METRIC in {"rmse", "mse", "mae"}
MAX_EPOCHS = 160
PATIENCE = 25
BATCH_SIZE = 256
# CPU выбран для воспроизводимого сравнения двух реализаций оптимизатора.
# Для большого запуска можно указать "cuda" или "mps", если устройство доступно.
DEVICE = torch.device("cpu")
torch.set_num_threads(2)


def seed_everything(seed=SEED):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)


seed_everything()
print(f"torch={torch.__version__}; numpy={np.__version__}; "
      f"pandas={pd.__version__}; sklearn={sklearn.__version__}; device={DEVICE}")
import json, pickle, statistics
from sklearn.decomposition import PCA
ARTIFACT_DIR = Path("/tmp/task4_combined")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

import torch.nn.functional as F
OUTPUT_DIR=ARTIFACT_DIR
RESULT_PATH=Path("/tmp/task4_combined_results.json")


torch=2.14.1; numpy=2.4.6; pandas=3.0.3; sklearn=1.9.0; device=cpu


In [2]:
candidates = [Path("task4_data"), Path("homeworks/task4_data")]
DATA_DIR = next((p for p in candidates if (p / "train_x.csv").is_file()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Запустите ноутбук из корня репозитория или из homeworks.")

train_x = pd.read_csv(DATA_DIR / "train_x.csv", index_col=0)
train_y = pd.read_csv(DATA_DIR / "train_y.csv", index_col=0)
test_table = pd.read_csv(DATA_DIR / "test_x.csv")
assert train_x.index.is_unique and train_y.index.is_unique
assert set(train_x.index) == set(train_y.index), "Индексы признаков и меток не совпадают"
train_y = train_y.loc[train_x.index]
assert list(train_y.columns) == ["year"]
assert "id" in test_table and test_table["id"].is_unique
assert not test_table["id"].isna().any()

feature_columns = list(train_x.columns)
assert len(feature_columns) == 90
assert set(test_table.columns) == set(feature_columns) | {"id"}
test_ids = test_table["id"].copy()
test_x = test_table.loc[:, feature_columns].copy()
assert all(pd.api.types.is_numeric_dtype(t) for t in train_x.dtypes)
assert all(pd.api.types.is_numeric_dtype(t) for t in test_x.dtypes)
X = train_x.to_numpy(dtype=np.float64)
X_test = test_x.to_numpy(dtype=np.float64)
y = torch.tensor(train_y["year"].to_numpy(dtype=np.float32))
assert torch.isfinite(y).all() and len(y) == len(X)
assert torch.equal(y, y.round()), "Годы должны быть целыми"
assert not np.isinf(X).any() and not np.isinf(X_test).any()

print(f"Обучение: {X.shape}; тест: {X_test.shape}; целевая переменная: year")
print(f"Пропуски: train={np.isnan(X).sum()}, test={np.isnan(X_test).sum()}")
print(f"Точные дубликаты train: {train_x.duplicated().sum()}")
for name in ("train_x.csv", "train_y.csv", "test_x.csv"):
    print(name, "SHA256:", hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest())

Обучение: (14000, 90); тест: (6000, 90); целевая переменная: year
Пропуски: train=0, test=0
Точные дубликаты train: 0
train_x.csv SHA256: aab2f64a4fdb0fd6af98ed95fedc9a3d9a54506b1f607da725926f654f5bff89
train_y.csv SHA256: 1f72519091e6e08948c430a1db9b181df28095e0cc9f29694570b40daed683bf
test_x.csv SHA256: 54f91d13c6f61fd634bae76bdad109471585a4ce0b308159be16f768134b4046


In [3]:
# NumPy/sklearn здесь применяются только к предварительному разбиению данных.
periods = np.digitize(y.tolist(), bins=[1970, 1980, 1990, 2000, 2005])
indices = np.arange(len(X))
development_idx, audit_idx = train_test_split(
    indices, test_size=0.15, random_state=SEED, stratify=periods
)
train_idx, valid_idx = train_test_split(
    development_idx, test_size=0.15 / 0.85, random_state=SEED,
    stratify=periods[development_idx]
)
assert not (set(train_idx) & set(valid_idx) or set(train_idx) & set(audit_idx)
            or set(valid_idx) & set(audit_idx))
assert len(set(train_idx) | set(valid_idx) | set(audit_idx)) == len(X)
row_hashes = pd.util.hash_pandas_object(train_x, index=False)
assert not (set(row_hashes.iloc[train_idx]) & set(row_hashes.iloc[valid_idx]))
assert not (set(row_hashes.iloc[development_idx]) & set(row_hashes.iloc[audit_idx]))
y_train, y_valid = y[train_idx], y[valid_idx]
print(f"train={len(train_idx)}, validation={len(valid_idx)}, audit={len(audit_idx)}")
print(f"Train: mean={y_train.mean():.2f}, std={y_train.std():.2f}, "
      f"min={y_train.min():.0f}, max={y_train.max():.0f}")

# audit_idx зафиксирован; ни предсказания, ни метрики audit здесь не вычисляются.


train=9800, validation=2100, audit=2100
Train: mean=1998.37, std=11.07, min=1922, max=2011


In [4]:
class SearchPreprocessor:
    def __init__(self, spec):
        self.spec = dict(spec)
        self.imputer = SimpleImputer(strategy="median", keep_empty_features=True)

    def _mask(self, d):
        block = self.spec.get("block", "all")
        return slice(0, 12) if block == "means" else (slice(12, d) if block == "cov" else slice(0, d))

    def fit(self, x):
        z = self.imputer.fit_transform(x)
        self.d = z.shape[1]
        kind = self.spec["kind"]
        self.robust = RobustScaler().fit(z)
        if kind == "winsor":
            q = self.spec["q"]
            self.bounds = np.quantile(z, [q, 1-q], axis=0)
        if kind in {"quantile", "hybrid"}:
            self.qt = QuantileTransformer(n_quantiles=min(self.spec.get("n_quantiles", 512), len(z)),
                        output_distribution=self.spec.get("distribution", "normal"),
                        subsample=None, random_state=42).fit(z[:, self._mask(self.d)])
        z = self._nonlinear(z)
        self.scaler = StandardScaler().fit(z)
        if kind == "pca":
            zz = self.scaler.transform(z)
            self.pca = PCA(n_components=self.spec.get("components", self.d), svd_solver="full").fit(zz)
            self.pca_scale = np.maximum(self.pca.explained_variance_, self.pca.explained_variance_.max()*0.01) ** (self.spec.get("whiten_power", 0)/2)
        if kind == "augment":
            self.aug_scaler = StandardScaler().fit(self._augment(self.scaler.transform(z)))
        return self

    def _nonlinear(self, z):
        z = z.copy()
        kind = self.spec["kind"]
        mask = self._mask(z.shape[1])
        if kind == "winsor":
            return np.clip(z, self.bounds[0], self.bounds[1])
        if kind in {"asinh", "signedlog", "robust"}:
            r = self.robust.transform(z)
            if kind == "robust": return r
            v = r[:, mask] / self.spec.get("scale", 1.)
            z[:, mask] = np.arcsinh(v) if kind == "asinh" else np.sign(v)*np.log1p(np.abs(v))
        if kind in {"quantile", "hybrid"}:
            z[:, mask] = self.qt.transform(z[:, mask])
        return z

    @staticmethod
    def _augment(z):
        extra = []
        for a in (z[:, :12], z[:, 12:]):
            extra.extend([np.log1p(np.sqrt(np.mean(a*a, axis=1))), np.log1p(np.mean(np.abs(a), axis=1))])
        return np.stack(extra, axis=1)

    def transform(self, x):
        z = self._nonlinear(self.imputer.transform(x))
        if self.spec["kind"] != "robust": z = self.scaler.transform(z)
        if self.spec["kind"] == "clip": z = np.clip(z, -self.spec["limit"], self.spec["limit"])
        if self.spec["kind"] == "pca": z = self.pca.transform(z)/self.pca_scale
        if self.spec["kind"] == "augment": z = np.concatenate([z, self.aug_scaler.transform(self._augment(z))], axis=1)
        assert np.isfinite(z).all()
        return torch.from_numpy(z.astype(np.float32))


In [5]:
class ExtendedPreprocessor(SearchPreprocessor):
    def __init__(self, spec):
        self.original_spec = dict(spec)
        self.engineered = spec["kind"] == "engineered"
        super().__init__({"kind":"standard"} if self.engineered else spec)

    def engineer(self, z):
        name = self.original_spec["feature"]
        if name == "means_quad":
            a = np.clip(z[:,:12],-5,5); ii,jj=np.triu_indices(12)
            return np.concatenate([z,a[:,ii]*a[:,jj]],axis=1)
        if name == "means_sq": return np.concatenate([z,np.clip(z[:,:12],-5,5)**2],axis=1)
        if name == "all_sq": return np.concatenate([z,np.clip(z,-5,5)**2],axis=1)
        if name.startswith("power"):
            p=float(name.removeprefix("power"));return np.sign(z)*np.abs(z)**p
        a,b=z[:,:12],z[:,12:]
        na=np.maximum(np.sqrt(np.mean(a*a,axis=1,keepdims=True)),1e-5)
        nb=np.maximum(np.sqrt(np.mean(b*b,axis=1,keepdims=True)),1e-5)
        if name == "blocknorm":return np.concatenate([a/na,b/nb,np.log1p(na),np.log1p(nb)],axis=1)
        if name == "blocknorm_mix":return np.concatenate([z,a/na,b/nb,np.log1p(na),np.log1p(nb)],axis=1)
        if name == "covnorm":return np.concatenate([a,b/nb,np.log1p(nb)],axis=1)
        if name == "meansnorm":return np.concatenate([a/na,b,np.log1p(na)],axis=1)
        raise ValueError(name)

    def fit(self,x):
        super().fit(x)
        if self.engineered:
            z=super().transform(x).numpy()
            self.engineer_scaler=StandardScaler().fit(self.engineer(z))
        return self

    def transform(self,x):
        z=super().transform(x)
        if self.engineered:
            z=torch.from_numpy(self.engineer_scaler.transform(self.engineer(z.numpy())).astype(np.float32))
        assert torch.isfinite(z).all()
        return z

# Идентичный цикл обучения; только класс preprocessing заменён на расширение.


In [6]:
class TargetScaler:
    def __init__(self, target):
        self.mean = float(target.mean())
        self.std = max(float(target.std(unbiased=False)), 1e-6)

    def encode(self, target):
        return (target - self.mean) / self.std

    def decode(self, prediction):
        return prediction * self.std + self.mean


@torch.inference_mode()
def regression_metrics(prediction, target):
    assert prediction.shape == target.shape
    error = prediction.double() - target.double()
    mse = error.square().mean().item()
    return {"rmse": math.sqrt(mse), "mse": mse, "mae": error.abs().mean().item()}


def print_metrics(label, values):
    print(f"{label}: RMSE={values['rmse']:.4f}; MSE={values['mse']:.4f}; MAE={values['mae']:.4f}")




In [7]:
class CustomAdamW(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, betas=(0.9, 0.999), eps=1e-8, weight_decay=1e-2):
        defaults = dict(lr=lr, betas=betas, eps=eps, weight_decay=weight_decay)
        super().__init__(params, defaults)
        for group in self.param_groups:
            if not math.isfinite(group["lr"]) or group["lr"] < 0:
                raise ValueError("lr должен быть конечным и неотрицательным")
            if not math.isfinite(group["eps"]) or group["eps"] <= 0:
                raise ValueError("eps должен быть конечным и положительным")
            if not math.isfinite(group["weight_decay"]) or group["weight_decay"] < 0:
                raise ValueError("weight_decay должен быть конечным и неотрицательным")
            if len(group["betas"]) != 2 or not all(0 <= b < 1 for b in group["betas"]):
                raise ValueError("betas должны лежать в [0, 1)")

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()
        for group in self.param_groups:
            beta1, beta2 = group["betas"]
            for parameter in group["params"]:
                grad = parameter.grad
                if grad is None:
                    continue
                if grad.is_sparse or torch.is_complex(parameter):
                    raise RuntimeError("Поддерживаются только плотные вещественные градиенты")
                state = self.state[parameter]
                if not state:
                    state["step"] = 0
                    state["exp_avg"] = torch.zeros_like(parameter)
                    state["exp_avg_sq"] = torch.zeros_like(parameter)
                state["step"] += 1
                m, v = state["exp_avg"], state["exp_avg_sq"]
                m.lerp_(grad, 1 - beta1)
                v.mul_(beta2).addcmul_(grad, grad, value=1 - beta2)
                correction1 = 1 - beta1 ** state["step"]
                correction2 = 1 - beta2 ** state["step"]
                denominator = v.sqrt().div_(math.sqrt(correction2)).add_(group["eps"])
                parameter.mul_(1 - group["lr"] * group["weight_decay"])
                parameter.addcdiv_(m, denominator, value=-group["lr"] / correction1)
        return loss

In [8]:
class TestCustomAdamW(unittest.TestCase):
    def test_reference_steps_and_parameter_groups(self):
        generator = torch.Generator().manual_seed(101)
        ours = [nn.Parameter(torch.randn(3, 4, generator=generator, dtype=torch.float64)),
                nn.Parameter(torch.randn(4, generator=generator, dtype=torch.float64))]
        reference = [nn.Parameter(p.detach().clone()) for p in ours]
        def groups(parameters):
            return [{"params": [parameters[0]], "lr": 0.003, "weight_decay": 0.1},
                    {"params": [parameters[1]], "lr": 0.002, "weight_decay": 0.0}]
        custom = CustomAdamW(groups(ours), betas=(0.8, 0.95), eps=1e-6)
        builtin = torch.optim.AdamW(groups(reference), betas=(0.8, 0.95), eps=1e-6,
                                    foreach=False, fused=False)
        for step in range(20):
            for i, (p, q) in enumerate(zip(ours, reference)):
                grad = torch.randn(p.shape, generator=generator, dtype=p.dtype)
                if step % 4 == 0 and i == 1:
                    p.grad = q.grad = None
                else:
                    p.grad = torch.zeros_like(grad) if step == 7 else grad
                    q.grad = p.grad.clone()
            custom.step()
            builtin.step()
            for p, q in zip(ours, reference):
                torch.testing.assert_close(p, q, rtol=1e-11, atol=1e-12)

    def test_none_vs_zero_gradient(self):
        p = nn.Parameter(torch.tensor([2.0], dtype=torch.float64))
        optimizer = CustomAdamW([p], lr=0.1, weight_decay=0.2)
        optimizer.step()
        self.assertEqual(p.item(), 2.0)
        self.assertNotIn(p, optimizer.state)
        p.grad = torch.zeros_like(p)
        optimizer.step()
        self.assertAlmostEqual(p.item(), 1.96)
        self.assertEqual(optimizer.state[p]["step"], 1)

    def test_restore_state(self):
        p = nn.Parameter(torch.tensor([1.0, -2.0], dtype=torch.float64))
        a = CustomAdamW([p])
        p.grad = torch.tensor([0.3, -0.1], dtype=p.dtype)
        a.step()
        q = nn.Parameter(p.detach().clone())
        b = CustomAdamW([q])
        b.load_state_dict(copy.deepcopy(a.state_dict()))
        for _ in range(5):
            p.grad = torch.tensor([-0.2, 0.4], dtype=p.dtype)
            q.grad = p.grad.clone()
            a.step()
            b.step()
        torch.testing.assert_close(p, q, rtol=0, atol=0)

    def test_invalid_settings(self):
        for kwargs in [{"lr": -1}, {"eps": 0}, {"betas": (1, 0.9)}, {"weight_decay": -1}]:
            with self.assertRaises(ValueError):
                CustomAdamW([nn.Parameter(torch.ones(1))], **kwargs)


suite = unittest.defaultTestLoader.loadTestsFromTestCase(TestCustomAdamW)
test_result = unittest.TextTestRunner(verbosity=2).run(suite)
assert test_result.wasSuccessful()

test_invalid_settings (__main__.TestCustomAdamW.test_invalid_settings) ... ok
test_none_vs_zero_gradient (__main__.TestCustomAdamW.test_none_vs_zero_gradient) ... ok
test_reference_steps_and_parameter_groups (__main__.TestCustomAdamW.test_reference_steps_and_parameter_groups) ... ok
test_restore_state (__main__.TestCustomAdamW.test_restore_state) ... ok

----------------------------------------------------------------------
Ran 4 tests in 0.018s

OK


In [9]:
class ResidualBlock(nn.Module):
    def __init__(self, width, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(width), nn.Linear(width, width * 2), nn.SiLU(),
            nn.Dropout(dropout), nn.Linear(width * 2, width), nn.Dropout(dropout)
        )
        nn.init.zeros_(self.net[-2].weight)
        nn.init.zeros_(self.net[-2].bias)

    def forward(self, x):
        return x + self.net(x)


class YearRegressor(nn.Module):
    def __init__(self, n_features, architecture="residual", width=128, dropout=0.15):
        super().__init__()
        if architecture == "linear":
            self.net = nn.Linear(n_features, 1)
        elif architecture == "mlp":
            self.net = nn.Sequential(
                nn.Linear(n_features, width), nn.ReLU(), nn.Dropout(dropout),
                nn.Linear(width, width), nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, 1)
            )
        elif architecture == "residual":
            self.net = nn.Sequential(
                nn.Linear(n_features, width), nn.SiLU(),
                ResidualBlock(width, dropout), ResidualBlock(width, dropout),
                nn.LayerNorm(width), nn.SiLU(), nn.Linear(width, 1)
            )
        else:
            raise ValueError(architecture)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def build_optimizer(model, kind, lr, weight_decay):
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    groups = [{"params": decay, "weight_decay": weight_decay},
              {"params": no_decay, "weight_decay": 0.0}]
    if kind == "custom":
        return CustomAdamW(groups, lr=lr)
    if kind == "builtin":
        return torch.optim.AdamW(groups, lr=lr, foreach=False, fused=False)
    raise ValueError(kind)


@torch.inference_mode()
def predict(model, features, target_scaler, batch_size=1024):
    model.eval()
    outputs = [model(batch.to(DEVICE)).cpu() for batch in features.split(batch_size)]
    return target_scaler.decode(torch.cat(outputs))


def make_loader(features, targets, seed):
    return DataLoader(TensorDataset(features, targets), batch_size=BATCH_SIZE, shuffle=True,
                      generator=torch.Generator().manual_seed(seed), num_workers=0)


def train_epoch(model, loader, optimizer):
    model.train()
    loss_fn = nn.L1Loss() if SELECTION_METRIC == "mae" else nn.MSELoss()
    total_loss, total_grad, count = 0.0, 0.0, 0
    for features, targets in loader:
        features, targets = features.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        prediction = model(features)
        assert prediction.shape == targets.shape
        loss = loss_fn(prediction, targets)
        if not torch.isfinite(loss):
            raise RuntimeError("Обучение разошлось")
        loss.backward()
        grad_norm = nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0,
                                           error_if_nonfinite=True)
        optimizer.step()
        total_loss += loss.item() * len(features)
        total_grad += float(grad_norm) * len(features)
        count += len(features)
    return total_loss / count, total_grad / count


def learning_rate(config, epoch):
    # Та же функция используется при final refit, без изменения горизонта.
    phase = (epoch - 1) / max(1, MAX_EPOCHS - 1)
    return config["lr"] * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * phase)))




## Одна сеть с вероятностным выходом

Сетка годов и prior строятся исключительно по train текущего разбиения. Сеть выдаёт 91 logit; к ним добавляется сглаженный эмпирический log-prior `(count+0.5)`. Softmax задаёт распределение, а сумма `probability × year` — прогноз. Гауссова мягкая разметка использует sigma0.6 года. MSE считается в нормированных годах, метрика выбора — MSE в исходных годах².

Код сохранён из сильного варианта `task4_objectives.ipynb`, включая порядок инициализации слоёв; единственное расширение — число входных признаков, чтобы поддержать добавленные признаки. Начальный контроль standard должен воспроизводить тот же результат.


In [10]:
class CombinedNetwork(nn.Module):
    def __init__(self, config, training_target):
        super().__init__()
        self.config = dict(config)
        self.kind = config["kind"]
        self.target_scaler = TargetScaler(training_target)
        self.mean = self.target_scaler.mean
        self.std = self.target_scaler.std
        self.low = float(training_target.min()) - config.get("margin", 0)
        self.high = float(training_target.max()) + config.get("margin", 0)
        self.register_buffer("support", torch.linspace(self.low, self.high, config.get("bins", 91)))
        if self.kind in {"distribution", "cdf"}:
            output_dim = len(self.support)
        elif self.kind == "ordinal":
            output_dim = len(self.support) - 1
        elif self.kind == "gaussian":
            output_dim = 2
        else:
            output_dim = 1
        if config.get("backbone", "residual") == "batchnorm":
            width, dropout = config["width"], config.get("bn_dropout", 0.4)
            self.net = nn.Sequential(nn.Linear(config["n_features"], width), nn.BatchNorm1d(width), nn.ReLU(),
                                     nn.Dropout(dropout), nn.Linear(width, width), nn.BatchNorm1d(width),
                                     nn.ReLU(), nn.Dropout(dropout), nn.Linear(width, 1))
        else:
            self.net = YearRegressor(config["n_features"], "residual", config["width"], config["dropout"]).net
        if output_dim != 1:
            self.net[-1] = nn.Linear(config["width"], output_dim)
        if self.kind in {"distribution", "cdf"} and config.get("prior", False):
            nearest = (training_target[:,None] - self.support[None,:]).abs().argmin(1)
            counts = torch.bincount(nearest, minlength=len(self.support)).float() + 0.5
            self.register_buffer("prior_logits", (counts/counts.sum()).log())
        else:
            self.register_buffer("prior_logits", torch.zeros(len(self.support)))

    def forward(self, features):
        raw = self.net(features)
        if self.kind in {"mse", "huber"}:
            z = raw[:,0]
        elif self.kind == "bounded":
            p = (self.mean-self.low)/(self.high-self.low)
            p = min(max(p, 1e-5), 1-1e-5)
            logit = math.log(p/(1-p))
            year = self.low + (self.high-self.low)*torch.sigmoid(raw[:,0]+logit)
            z = (year-self.mean)/self.std
        elif self.kind == "upper_softplus":
            offset = math.log(math.expm1(max((self.high-self.mean)/self.std, 1e-3)))
            z = (self.high-self.mean)/self.std-F.softplus(raw[:,0]+offset)
        elif self.kind == "gaussian":
            z = raw[:,0]
        elif self.kind in {"distribution", "cdf"}:
            probs = (raw+self.prior_logits).softmax(1)
            z = ((probs*self.support).sum(1)-self.mean)/self.std
        elif self.kind == "ordinal":
            probs = torch.sigmoid(raw)
            monotone = torch.cummin(probs, dim=1).values
            step = (self.high-self.low)/(len(self.support)-1)
            z = (self.low+step*monotone.sum(1)-self.mean)/self.std
        else:
            raise ValueError(self.kind)
        return z, raw

    def loss(self, prediction, raw, target):
        mse = F.mse_loss(prediction, target)
        c = self.config
        if self.kind == "huber":
            return F.huber_loss(prediction, target, delta=c["delta"])
        if self.kind == "gaussian":
            logvar = raw[:,1].clamp(-3, 3)
            nll = 0.5*((prediction-target).square()*torch.exp(-logvar)+logvar).mean()
            return c["mse_weight"]*mse+c["aux_weight"]*nll
        if self.kind in {"distribution", "cdf"}:
            years = target*self.std+self.mean
            sigma = c.get("sigma", 1.0)
            target_probs = (-0.5*((years[:,None]-self.support[None,:])/sigma).square()).softmax(1)
            logp = (raw+self.prior_logits).log_softmax(1)
            if self.kind == "distribution":
                auxiliary = -(target_probs*logp).sum(1).mean()
            else:
                auxiliary = F.mse_loss(logp.exp().cumsum(1), target_probs.cumsum(1))
            return c["mse_weight"]*mse+c["aux_weight"]*auxiliary
        if self.kind == "ordinal":
            years = target*self.std+self.mean
            thresholds = 0.5*(self.support[:-1]+self.support[1:])
            target_survival = (years[:,None]>thresholds[None,:]).float()
            auxiliary = F.binary_cross_entropy_with_logits(raw, target_survival)
            return c["mse_weight"]*mse+c["aux_weight"]*auxiliary
        return mse

@torch.inference_mode()
def objective_predict(model, features):
    model.eval()
    z = torch.cat([model(batch)[0] for batch in features.split(1024)])
    return z*model.std+model.mean



In [11]:
def make_split(seed):
    return train_test_split(development_idx,test_size=0.15/0.85,random_state=seed,
                             stratify=periods[development_idx])

def run_combined(config,seed=42,optimizer_kind="builtin"):
    seed_everything(seed)
    tr,va=make_split(seed)
    pp=ExtendedPreprocessor(config['preprocessing']).fit(X[tr])
    xt,xv=pp.transform(X[tr]),pp.transform(X[va])
    actual_config=dict(config,n_features=xt.shape[1])
    target_scaler=TargetScaler(y[tr])
    loader=make_loader(xt,target_scaler.encode(y[tr]),seed)
    model=CombinedNetwork(actual_config,y[tr])
    optimizer=build_optimizer(model,optimizer_kind,config['lr'],config['weight_decay'])
    best_score,best_epoch,best_state,stale=math.inf,0,None,0
    start=time.perf_counter();history=[]
    for epoch in range(1,161):
        model.train()
        for group in optimizer.param_groups:group['lr']=learning_rate(config,epoch)
        loss_sum=grad_sum=0.
        for xb,yb in loader:
            optimizer.zero_grad(set_to_none=True)
            pred,raw=model(xb);loss=model.loss(pred,raw,yb)
            assert torch.isfinite(loss)
            loss.backward()
            grad=nn.utils.clip_grad_norm_(model.parameters(),5.,error_if_nonfinite=True)
            optimizer.step()
            loss_sum+=float(loss.detach())*len(xb);grad_sum+=float(grad)*len(xb)
        prediction=objective_predict(model,xv)
        metrics=regression_metrics(prediction,y[va])
        history.append(dict(epoch=epoch,loss=loss_sum/len(tr),grad_norm=grad_sum/len(tr),**metrics))
        if metrics['mse']<best_score:
            best_score,best_epoch,stale=metrics['mse'],epoch,0
            best_state=copy.deepcopy(model.state_dict())
        else:stale+=1
        if stale>=25:break
    model.load_state_dict(best_state)
    prediction=objective_predict(model,xv)
    metrics=regression_metrics(prediction,y[va])
    rid=f"{config['id']}_s{seed}_{optimizer_kind}"
    path=OUTPUT_DIR/f'{rid}.pt'
    record=dict(id=rid,config=actual_config,split_seed=seed,optimizer=optimizer_kind,best_epoch=best_epoch,
                history=history,metrics=metrics,checkpoint=str(path),seconds=time.perf_counter()-start)
    torch.save(dict(state_dict=best_state,config=actual_config,seed=seed,best_epoch=best_epoch,
                    train_idx=torch.tensor(tr),valid_idx=torch.tensor(va),valid_prediction=prediction,
                    target_mean=model.mean,target_std=model.std),path)
    with open(OUTPUT_DIR/f'{rid}_preprocessing.pkl','wb') as f:pickle.dump(pp,f)
    print(f"{rid:42s} MSE={metrics['mse']:.6f} RMSE={metrics['rmse']:.6f} epoch={best_epoch} seconds={record['seconds']:.1f}",flush=True)
    return record


In [12]:
common=dict(width=128,dropout=.15,lr=.001,weight_decay=.1,kind="distribution",bins=91,
            sigma=.6,prior=True,mse_weight=1,aux_weight=2,backbone="batchnorm",bn_dropout=.55)
specs=[("standard",dict(kind="standard")),("pca0",dict(kind="pca",whiten_power=0.)),
       ("pca0.5",dict(kind="pca",whiten_power=.5)),("meansnorm",dict(kind="engineered",feature="meansnorm")),
       ("meansquad",dict(kind="engineered",feature="means_quad")),("rowscale",dict(kind="augment")),
       ("clip3",dict(kind="clip",limit=3.)),("quantile_means",dict(kind="hybrid",block="means")),
       ("clip5",dict(kind="clip",limit=5.)),("asinh_means",dict(kind="asinh",block="means")),
       ("blocknorm",dict(kind="engineered",feature="blocknorm")),("pca70",dict(kind="pca",components=70,whiten_power=.5))]
configs=[dict(common,id=f'combined_{name}',preprocessing=spec) for name,spec in specs]
assert len(configs)==12
results=[]
for config in configs:
    results.append(run_combined(config))
    RESULT_PATH.write_text(json.dumps(results,indent=2))
print('Ранжирование12 вариантов:')
for r in sorted(results,key=lambda r:r['metrics']['mse']):print(r['config']['id'],r['metrics'],'epoch',r['best_epoch'])
# This assertion checks literal reproduction of the upstream architecture/head control.
standard_record=next(r for r in results if r['config']['id']=='combined_standard')
assert abs(standard_record['metrics']['mse']-79.30734367)<1e-3
leader_ids=[r['config']['id'] for r in sorted(results,key=lambda r:r['metrics']['mse'])[:3]]
confirm_ids=list(dict.fromkeys(['combined_standard']+leader_ids))
print('Фиксируем перед подтверждением:',confirm_ids)
for cid in confirm_ids:
    cfg=next(c for c in configs if c['id']==cid)
    for seed in [17,83]:
        results.append(run_combined(cfg,seed))
        RESULT_PATH.write_text(json.dumps(results,indent=2))
comparison=[]
for cid in confirm_ids:
    rr=[r for r in results if r['config']['id']==cid]
    comparison.append(dict(id=cid,mean_mse=statistics.mean(r['metrics']['mse'] for r in rr),
                       mse_by_seed={str(r['split_seed']):r['metrics']['mse'] for r in rr},
                       refit_epochs=int(statistics.median(r['best_epoch'] for r in rr))))
comparison.sort(key=lambda r:r['mean_mse'])
print('Подтверждённые результаты:')
for r in comparison:print(r)
(OUTPUT_DIR/'comparison.json').write_text(json.dumps(comparison,indent=2))


combined_standard_s42_builtin              MSE=79.307344 RMSE=8.905467 epoch=20 seconds=3.2
combined_pca0_s42_builtin                  MSE=79.969636 RMSE=8.942574 epoch=31 seconds=3.9
combined_pca0.5_s42_builtin                MSE=80.072268 RMSE=8.948311 epoch=31 seconds=4.0
combined_meansnorm_s42_builtin             MSE=79.327719 RMSE=8.906611 epoch=40 seconds=4.7
combined_meansquad_s42_builtin             MSE=78.441632 RMSE=8.856728 epoch=41 seconds=4.9
combined_rowscale_s42_builtin              MSE=80.043493 RMSE=8.946703 epoch=32 seconds=4.2
combined_clip3_s42_builtin                 MSE=79.905729 RMSE=8.939000 epoch=20 seconds=3.3
combined_quantile_means_s42_builtin        MSE=79.737623 RMSE=8.929593 epoch=20 seconds=3.2
combined_clip5_s42_builtin                 MSE=79.359325 RMSE=8.908385 epoch=31 seconds=4.1
combined_asinh_means_s42_builtin           MSE=79.547544 RMSE=8.918943 epoch=20 seconds=3.3
combined_blocknorm_s42_builtin             MSE=80.066286 RMSE=8.947977 epoch=29 

## Финальные кандидаты и собственный AdamW

По среднему MSE трёх разбиений сохраняются до трёх лучших альтернатив. Каждый файл CSV производится отдельной сетью, обученной на всех 14000 строках. Число эпох фиксируется медианой validation epochs, learning-rate horizon160 сохраняется. Лучший кандидат дополнительно сравнивается со встроенным AdamW при одинаковом seed. Public/private score не выводится из validation.


In [13]:
submission_ids=[r['id'] for r in comparison[:3]]
best_config=next(c for c in configs if c['id']==submission_ids[0])
custom=run_combined(best_config,42,'custom')
results.append(custom);RESULT_PATH.write_text(json.dumps(results,indent=2))
original=next(r for r in results if r['config']['id']==submission_ids[0] and r['split_seed']==42 and r['optimizer']=='builtin')
a=torch.load(original['checkpoint'],weights_only=False,map_location='cpu')
b=torch.load(custom['checkpoint'],weights_only=False,map_location='cpu')
max_difference=float((a['valid_prediction']-b['valid_prediction']).abs().max())
print('Custom minus builtin MSE:',custom['metrics']['mse']-original['metrics']['mse'])
print('Maximum prediction difference:',max_difference)
assert max_difference<1e-4


combined_meansquad_s42_custom              MSE=78.441632 RMSE=8.856728 epoch=41 seconds=4.4
Custom minus builtin MSE: 0.0
Maximum prediction difference: 0.0


In [14]:
submission_dir=DATA_DIR/'Submission';submission_dir.mkdir(exist_ok=True)
final_records=[]
for cid in submission_ids:
    config=next(c for c in configs if c['id']==cid)
    epochs=next(r['refit_epochs'] for r in comparison if r['id']==cid)
    seed_everything(42)
    pp=ExtendedPreprocessor(config['preprocessing']).fit(X)
    xt=pp.transform(X)
    actual_config=dict(config,n_features=xt.shape[1])
    scaler=TargetScaler(y)
    loader=make_loader(xt,scaler.encode(y),42)
    model=CombinedNetwork(actual_config,y)
    optimizer=build_optimizer(model,'custom',config['lr'],config['weight_decay'])
    history=[]
    for epoch in range(1,epochs+1):
        model.train();loss_sum=0.
        for group in optimizer.param_groups:group['lr']=learning_rate(config,epoch)
        for xb,yb in loader:
            optimizer.zero_grad(set_to_none=True)
            pred,raw=model(xb);loss=model.loss(pred,raw,yb)
            assert torch.isfinite(loss)
            loss.backward();nn.utils.clip_grad_norm_(model.parameters(),5.,error_if_nonfinite=True)
            optimizer.step();loss_sum+=float(loss.detach())*len(xb)
        history.append(dict(epoch=epoch,loss=loss_sum/len(X)))
    prediction=objective_predict(model,pp.transform(X_test))
    assert prediction.shape==(6000,) and torch.isfinite(prediction).all()
    submission=pd.DataFrame({'id':test_ids.to_numpy(),'year':prediction.numpy()})
    csv_path=submission_dir/f'task4_{cid}.csv'
    submission.to_csv(csv_path,index=False,float_format='%.6f')
    actual=pd.read_csv(csv_path)
    assert list(actual.columns)==['id','year'] and len(actual)==6000
    assert actual['id'].equals(test_ids.reset_index(drop=True))
    torch.testing.assert_close(torch.tensor(actual['year'].to_numpy()),prediction.double(),rtol=0,atol=1e-6)
    checkpoint=OUTPUT_DIR/f'final_{cid}.pt'
    torch.save(dict(state_dict=model.state_dict(),config=actual_config,target_mean=model.mean,
                    target_std=model.std,epochs=epochs,history=history,test_prediction=prediction),checkpoint)
    with open(OUTPUT_DIR/f'final_{cid}_preprocessing.pkl','wb') as f:pickle.dump(pp,f)
    record=dict(id=cid,epochs=epochs,csv=str(csv_path),checkpoint=str(checkpoint),
                sha256=hashlib.sha256(csv_path.read_bytes()).hexdigest(),
                prediction_min=float(prediction.min()),prediction_max=float(prediction.max()))
    final_records.append(record);print(record)
for run in results:
    cp=torch.load(run['checkpoint'],weights_only=False,map_location='cpu')
    assert not (set(cp['train_idx'].tolist())|set(cp['valid_idx'].tolist()))&set(audit_idx.tolist())
baseline_path=Path('homeworks/task4.ipynb') if Path('homeworks/task4.ipynb').is_file() else Path('task4.ipynb')
assert hashlib.sha256(baseline_path.read_bytes()).hexdigest()=='d3883b0f7909703c335dcdd44e252ef8101d0fea16b3b0da1fa3b33807735419'
print('CSV проверены. Исходныйtask4.ipynb неизменён; audit не оценивался.')
final_report=dict(runs=results,comparison=comparison,submissions=final_records,
                  custom_max_prediction_difference=max_difference,audit_evaluated=False)
(OUTPUT_DIR/'final_report.json').write_text(json.dumps(final_report,indent=2))


{'id': 'combined_meansquad', 'epochs': 30, 'csv': 'homeworks/task4_data/Submission/task4_combined_meansquad.csv', 'checkpoint': '/tmp/task4_combined/final_combined_meansquad.pt', 'sha256': '3bf3253cc4c6acdec0cc9ea2a92a34ae2a32c6c94c5f25f01bb9921d43fd7b1d', 'prediction_min': 1975.6151123046875, 'prediction_max': 2007.8046875}
{'id': 'combined_meansnorm', 'epochs': 33, 'csv': 'homeworks/task4_data/Submission/task4_combined_meansnorm.csv', 'checkpoint': '/tmp/task4_combined/final_combined_meansnorm.pt', 'sha256': '30122b89b23ca7b4de36e5800a7024a0dd19f623052da33f4c58120ec59a9630', 'prediction_min': 1977.322021484375, 'prediction_max': 2006.79443359375}
{'id': 'combined_standard', 'epochs': 31, 'csv': 'homeworks/task4_data/Submission/task4_combined_standard.csv', 'checkpoint': '/tmp/task4_combined/final_combined_standard.pt', 'sha256': 'a1b684188bde0829dbb5192a1144bba887d20a3588760930b4484b16e72d8935', 'prediction_min': 1977.4212646484375, 'prediction_max': 2006.9361572265625}
CSV проверены

Лучший объединённый вариант по трём development-разбиениям — `combined_meansquad`: MSE78.397123 против 78.757411 у стандартной предобработки. Это небольшое дополнительное улучшение 0.3603; общий прогресс относительно исходной residual-сети 83.5600 существенно больше.

## Интерпретация

Верхняя таблица сравнивает preprocessing уже при фиксированной сильной вероятностной голове. При равном качестве более простая стандартная предобработка предпочтительнее сложной инженерии признаков. Разница нескольких десятых MSE сама по себе не доказывает превосходство: splits пересекаются, и множество конфигураций выбрано по этим данным. Финальный вывод требует отдельно зафиксированной контрольной проверки; никаких новых гипотез после audit здесь не планируется.

Источники: локальные условия `homeworks/pytorch.md`, семинар `notebooks/pytorch_intro.ipynb`, базовое решение `task4.ipynb` и собственные эксперименты `task4_preprocessing.ipynb` / `task4_objectives.ipynb`. Обучение не читает их файлы и не зависит от их выполнения. Формула оптимизатора: [Loshchilov & Hutter](https://arxiv.org/abs/1711.05101), API: [PyTorch AdamW](https://docs.pytorch.org/docs/stable/generated/torch.optim.AdamW.html). Внешние данные не использованы.


## Приложение: однократная диагностика на audit после фиксации всех решений

Все семейства, гиперпараметры, shortlist и число эпох финального обучения зафиксированы до этой проверки. Audit2100 строк проверяется на checkpoint seed42, обученном только на 9800train строках, а не на модели после final refit. Никакие параметры и submission после просмотра этих чисел не меняются.

Audit исходного baseline уже был показан в предыдущем решении. В данном поиске он не использовался для ранней остановки или настройки. Поэтому это дополнительная диагностическая выборка, но не полностью ранее невиданный внешний тест. Главный кандидат `combined_meansquad` выбран по трём development-разбиениям и остаётся выбранным независимо от audit.


In [15]:
audit_records=[];audit_predictions={}
for cid in ['combined_meansquad','combined_meansnorm','combined_standard']:
    prefix=OUTPUT_DIR/f'{cid}_s42_builtin'
    cp=torch.load(str(prefix)+'.pt',weights_only=False,map_location='cpu')
    with open(str(prefix)+'_preprocessing.pkl','rb') as f:pp=pickle.load(f)
    model=CombinedNetwork(cp['config'],y[cp['train_idx']])
    model.load_state_dict(cp['state_dict'])
    prediction=objective_predict(model,pp.transform(X[audit_idx]))
    metric=regression_metrics(prediction,y[audit_idx])
    record=dict(id=cid,config=cp['config'],seed=42,training_rows=9800,audit_rows=2100,metrics=metric)
    audit_records.append(record);audit_predictions[cid]=prediction
    print(json.dumps(record,ensure_ascii=False))
Path('/tmp/task4_combined_audit.json').write_text(json.dumps(audit_records,indent=2))
torch.save(dict(audit_idx=torch.tensor(audit_idx),predictions=audit_predictions,records=audit_records),'/tmp/task4_combined_audit_predictions.pt')
report=json.loads((OUTPUT_DIR/'final_report.json').read_text())
report.update(audit_evaluated=True,audit_diagnostic=audit_records,audit_used_for_selection=False)
(OUTPUT_DIR/'final_report.json').write_text(json.dumps(report,indent=2))
print('Основной кандидатcombined_meansquad остаётся выбранным по development. Послеaudit ничего не настраивается.')


{"id": "combined_meansquad", "config": {"width": 128, "dropout": 0.15, "lr": 0.001, "weight_decay": 0.1, "kind": "distribution", "bins": 91, "sigma": 0.6, "prior": true, "mse_weight": 1, "aux_weight": 2, "backbone": "batchnorm", "bn_dropout": 0.55, "id": "combined_meansquad", "preprocessing": {"kind": "engineered", "feature": "means_quad"}, "n_features": 168}, "seed": 42, "training_rows": 9800, "audit_rows": 2100, "metrics": {"rmse": 9.148296933791112, "mse": 83.69133678881185, "mae": 6.161216692243303}}
{"id": "combined_meansnorm", "config": {"width": 128, "dropout": 0.15, "lr": 0.001, "weight_decay": 0.1, "kind": "distribution", "bins": 91, "sigma": 0.6, "prior": true, "mse_weight": 1, "aux_weight": 2, "backbone": "batchnorm", "bn_dropout": 0.55, "id": "combined_meansnorm", "preprocessing": {"kind": "engineered", "feature": "meansnorm"}, "n_features": 91}, "seed": 42, "training_rows": 9800, "audit_rows": 2100, "metrics": {"rmse": 8.988325413860101, "mse": 80.78999374544337, "mae": 6.